# Практическое задание. Офлайн-генерация описаний товаров

In [ ]:
import asyncio
import copy
import gc
import random
import re
import textwrap
import time
from collections import Counter
from dataclasses import dataclass
from math import log
from string import Template
from typing import List

import aiohttp
import nest_asyncio
import pandas as pd
import torch
from datasets import load_dataset
from difflib import SequenceMatcher
from vllm import LLM, SamplingParams

# Фиксируем seed
SEED = 42
random.seed(SEED)

In [ ]:
# Загружаем датасет
dataset = load_dataset("UniqueData/asos-e-commerce-dataset")

# Выбираем split (обычно 'train')
data = dataset['train']  # или другой доступный split

# Выбираем 100 случайных примеров
random_indices = random.sample(range(len(data)), 100)
random_samples = data.select(random_indices)

target_features = ["name", "size", "category", "price", "color"]
result_data = []
for sample in random_samples:
  target_features_sample = {feature_name: sample.get(feature_name) for feature_name in target_features}
  result_data.append(target_features_sample)

df = pd.DataFrame(result_data)

In [ ]:
# Посмотрим на данные
df.sample(5)

,name,size,category,price,color
25,ASOS Daysocial unisex co-ord relaxed joggers w...,"2XS,XS,S,M,L,XL,2XL - Out of stock",ASOS Daysocial unisex co-ord relaxed joggers w...,30.00,BLACK
51,Topshop Quilted Jacket in Ecru,"UK 4 - Out of stock,UK 6 - Out of stock,UK 8 -...",Topshop Quilted Jacket in Ecru,Now 27.95,White1
26,ASOS DESIGN Curve plisse midi wrap dress with ...,"UK 16,UK 18,UK 20,UK 22,UK 24,UK 26,UK 28,UK 30",ASOS DESIGN Curve plisse midi wrap dress with ...,36.00,ROSE
79,Monki teddy coat in brown,"2XS - UK 0-2,XS - UK 4-6,S - UK 8-10,M - UK 12...",Monki teddy coat in brown,Now 59.50,Brown
71,ASOS DESIGN tank top with front strap detail i...,"UK 4,UK 6,UK 8,UK 10,UK 12,UK 14,UK 16,UK 18",ASOS DESIGN tank top with front strap detail i...,Now 13.50,Cobalt


## Оффлайн генерация

In [ ]:
def check_h1_tags(description):
    """
    Проверяет наличие ровно одного непустого заголовка в тегах <h1>
    Возвращает 1 — если есть, 0 — если нет
    """
    h1_pattern = r'<h1>(.*?)</h1>'
    matches = re.findall(h1_pattern, description, re.DOTALL | re.IGNORECASE)

    # Должен быть ровно один непустой заголовок
    if len(matches) != 1:
        return 0

    # Проверяем, что заголовок не пустой и не состоит только из пробелов
    h1_content = matches[0].strip()
    if not h1_content or len(h1_content) < 3:
        return 0

    return 1

def similarity_ratio(a, b):
    """Вычисляет коэффициент схожести строк (0.0-1.0)"""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def calculate_lexical_diversity(text):
    """
    Вычисляет лексическое разнообразие с учётом уникальности слов
    и штрафует за повторяющиеся последовательности
    """
    # Удаляем HTML-теги
    clean_text = re.sub(r'<[^>]+>', '', text)
    words = re.findall(r'\b\w+\b', clean_text.lower())

    if len(words) < 5:  # Минимальное количество слов для анализа
        return 0.0

    # Подсчитываем частоту слов
    word_counts = Counter(words)
    total_words = len(words)
    unique_words = len(word_counts)

    # Базовая метрика разнообразия
    base_diversity = unique_words / total_words

    # Штраф за повторяющиеся последовательности
    repeat_penalty = 1.0
    for i in range(len(words) - 3):
        sequence = ' '.join(words[i:i+3])
        if sequence in ' '.join(words[i+3:]):
            repeat_penalty *= 0.7  # Штраф за повторение последовательностей

    # Комбинированная метрика
    diversity = base_diversity * repeat_penalty
    return min(diversity, 1.0)

def check_paragraph_formatting(description):
    """
    Строгая проверка форматирования абзацев
    """
    p_pattern = r'<p>(.*?)</p>'
    paragraphs = re.findall(p_pattern, description, re.DOTALL | re.IGNORECASE)

    if not paragraphs:
        return 0.0

    valid_count = 0
    for paragraph in paragraphs:
        # Убираем теги и проверяем содержание
        clean_content = re.sub(r'<[^>]+>', '', paragraph).strip()

        # Абзац должен содержать значимый текст (не только теги)
        if (len(clean_content) >= 10 and  # Минимальная длина
            not clean_content.isdigit() and  # Не только цифры
            len(set(clean_content.split())) >= 3):  # Минимум 3 уникальных слова
            valid_count += 1

    return valid_count / len(paragraphs) if paragraphs else 0.0

def is_characteristic_present(text, char_name, char_value, threshold=0.6):
    """
    Более строгая проверка наличия характеристики
    """
    text_lower = text.lower()
    char_name_lower = char_name.lower()
    char_value_lower = str(char_value).lower()

    # Проверяем полное совпадение или значимое частичное
    if (char_name_lower in text_lower and char_value_lower in text_lower):
        return True

    # Проверяем семантическую близость с более высоким порогом
    name_similarity = similarity_ratio(char_name_lower, text_lower)
    value_similarity = similarity_ratio(char_value_lower, text_lower)

    # Требуем высокой схожести для обеих частей характеристики
    return (name_similarity >= threshold and value_similarity >= threshold)

def check_paragraph_tags(description, characteristics, threshold=0.6):
    """
    Строгая проверка абзацев: один абзац = одна характеристика
    """
    if not characteristics:
        return 1.0

    p_pattern = r'<p>(.*?)</p>'
    paragraphs = re.findall(p_pattern, description, re.DOTALL | re.IGNORECASE)

    if not paragraphs:
        return 0.0

    # Штрафуем за дубликаты абзацев
    unique_paragraphs = set()
    duplicate_penalty = 1.0

    for paragraph in paragraphs:
        clean_text = re.sub(r'<[^>]+>', '', paragraph).strip()
        if clean_text in unique_paragraphs:
            duplicate_penalty *= 0.5  # Сильный штраф за дубликаты
        unique_paragraphs.add(clean_text)

    # Сопоставляем характеристики с абзацами
    char_matched = {char: False for char in characteristics}
    valid_matches = 0

    for paragraph in paragraphs:
        clean_text = re.sub(r'<[^>]+>', '', paragraph).strip()

        # Ищем характеристику в абзаце
        matched_chars = []
        for char_name, char_value in characteristics.items():
            if is_characteristic_present(clean_text, char_name, char_value, threshold):
                matched_chars.append(char_name)

        # Абзац должен содержать ровно одну характеристику
        if len(matched_chars) == 1 and not char_matched.get(matched_chars[0], False):
            char_matched[matched_chars[0]] = True
            valid_matches += 1

    # Учитываем штраф за дубликаты
    coverage_score = sum(1 for matched in char_matched.values() if matched) / len(characteristics)
    return coverage_score * duplicate_penalty

def check_all_characteristics_covered(description, characteristics, threshold=0.5):
    """
    Проверяет coverage характеристик с учётом уникальности
    """
    if not characteristics:
        return 1.0

    clean_description = re.sub(r'<[^>]+>', '', description)
    found_chars = set()

    for char_name, char_value in characteristics.items():
        if is_characteristic_present(clean_description, char_name, char_value, threshold):
            found_chars.add(char_name)

    return len(found_chars) / len(characteristics)

def check_text_quality(description):
    """
    Новая функция: проверяет общее качество текста
    """
    clean_text = re.sub(r'<[^>]+>', '', description)
    words = clean_text.split()

    if len(words) < 10:
        return 0.0

    # Проверяем разнообразие (повторы, уникальность)
    word_counts = Counter(words)
    unique_ratio = len(word_counts) / len(words)

    # Штрафуем за слишком частые повторы
    repeat_penalty = 1.0
    for word, count in word_counts.items():
        if count > len(words) * 0.1:  # Слово повторяется более 10% текста
            repeat_penalty *= 0.7

    return min(unique_ratio * repeat_penalty, 1.0)

def calculate_total_score(description, characteristics):
    """
    Пересмотренная система оценки с более строгими критериями
    """
    # Выполняем все проверки
    h1_score = check_h1_tags(description)
    paragraph_score = check_paragraph_tags(description, characteristics, threshold=0.7)
    coverage_score = check_all_characteristics_covered(description, characteristics, threshold=0.6)
    diversity_score = calculate_lexical_diversity(description)
    formatting_score = check_paragraph_formatting(description)
    text_quality_score = check_text_quality(description)

    # Новые весовые коэффициенты (увеличиваем вес за качество)
    weights = {
        'h1': 0.10,
        'paragraphs': 0.25,
        'coverage': 0.15,
        'diversity': 0.20,
        'formatting': 0.15,
        'quality': 0.15  # Новый критерий качества текста
    }

    total_score = (
        h1_score * weights['h1'] +
        paragraph_score * weights['paragraphs'] +
        coverage_score * weights['coverage'] +
        diversity_score * weights['diversity'] +
        formatting_score * weights['formatting'] +
        text_quality_score * weights['quality']
    ) * 100

    return {
        'total_score': round(total_score, 2),
        'detailed_scores': {
            'h1_present': h1_score,
            'paragraphs_validation': round(paragraph_score, 2),
            'characteristics_coverage': round(coverage_score, 2),
            'lexical_diversity': round(diversity_score, 2),
            'paragraph_formatting': round(formatting_score, 2),
            'text_quality': round(text_quality_score, 2)
        }
    }

# Пример использования
test_description = """
    <h1>Смартфон Premium X</h1>
    <p>Данная модель обладает превосходным дисплеем размером 6.7 дюймов.</p>
    <p>Основная камера 108 мегапикселей обеспечивает отличное качество фото.
    <p>Процессор Snapdragon 8 второго поколения обеспечивает высокую производительность.</p>
    """

test_characteristics = {
        'экран': '6.7 дюймов',
        'камера': '108 Мп',
        'батарея': '5000 мАч',
        'процессор': 'Snapdragon 8 Gen 2'
    }

result = calculate_total_score(test_description, test_characteristics)
print("Общий скор:", result['total_score'])
print("Детальные результаты:", result['detailed_scores'])

Общий скор: 58.73
Детальные результаты: {'h1_present': 1, 'paragraphs_validation': 0.0, 'characteristics_coverage': 0.0, 'lexical_diversity': 0.96, 'paragraph_formatting': 1.0, 'text_quality': 0.96}


In [ ]:
@dataclass
class SampleGenerationResult():
  description: str
  latency: float
  total_score: float
  detailed_score: dict

@dataclass
class FullGenerationResult():
  throughput: float
  latency_avg: float
  total_score_avg: float
  outputs: List[SampleGenerationResult]

def print_compact_stats(results: FullGenerationResult) -> None:
    """
    Компактная версия вывода статистики.
    """
    print(" СТАТИСТИКА ГЕНЕРАЦИИ")
    print(f"   Throughput: {results.throughput:.2f} req/sec")
    print(f"   Avg Latency: {results.latency_avg:.3f} sec")
    print(f"   Samples: {len(results.outputs)}")
    print(f"   Avg Score: {sum(s.total_score for s in results.outputs)/len(results.outputs):.3f}")

    # Добавляем случайные описания в компактную версию
    if len(results.outputs) >= 3:
        print("\n 3 случайных описания:")
        random_indices = random.sample(range(len(results.outputs)), min(3, len(results.outputs)))
        for i, idx in enumerate(random_indices, 1):
            desc = results.outputs[idx].description
            print(f"   {i}. {desc}")

In [ ]:
# Удаление модели и освобождение памяти
# del llm
gc.collect()

# Дополнительная очистка GPU памяти (если используется CUDA)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

model_name = "Qwen/Qwen3-30B-A3B-Instruct-2507"

# Инициализация модели
print("Загрузка модели...")
llm = LLM(
    model=model_name,
    tensor_parallel_size=1,
    max_model_len=20_000
)

INFO 04-18 17:55:38 [__init__.py:216] Automatically detected platform cuda.
Загрузка модели...
INFO 04-18 17:55:39 [utils.py:328] non-default args: {'max_model_len': 20000, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-30B-A3B-Instruct-2507'}
INFO 04-18 17:55:46 [__init__.py:742] Resolved architecture: Qwen3MoeForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 04-18 17:55:46 [__init__.py:1815] Using max model len 20000


2026-04-18 17:55:46,242	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 04-18 17:55:46 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 04-18 17:55:50 [__init__.py:2974] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 04-18 17:55:53 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=3507) INFO 04-18 17:55:54 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=3507) INFO 04-18 17:55:54 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: model='Qwen/Qwen3-30B-A3B-Instruct-2507', speculative_config=None, tokenizer='Qwen/Qwen3-30B-A3B-Instruct-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=20000, download_dir=None, load_format=auto, tensor_parallel_size=1, 

...

[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=3507) INFO 04-18 17:55:56 [parallel_state.py:1165] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=3507) WARNING 04-18 17:55:56 [topk_topp_sampler.py:69] FlashInfer is not available. Falling back to the PyTorch-native implementation of top-p & top-k sampling. For the best performance, please install FlashInfer.
(EngineCore_DP0 pid=3507) INFO 04-18 17:55:56 [gpu_model_runner.py:2

Loading safetensors checkpoint shards:   0% Completed | 0/16 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   6% Completed | 1/16 [00:00<00:11,  1.27it/s]
Loading safetensors checkpoint shards:  12% Completed | 2/16 [00:01<00:12,  1.16it/s]
Loading safetensors checkpoint shards:  19% Completed | 3/16 [00:02<00:11,  1.13it/s]
Loading safetensors checkpoint shards:  25% Completed | 4/16 [00:03<00:10,  1.10it/s]
Loading safetensors checkpoint shards:  31% Completed | 5/16 [00:04<00:10,  1.10it/s]
Loading safetensors checkpoint shards:  38% Completed | 6/16 [00:05<00:09,  1.10it/s]
Loading safetensors checkpoint shards:  44% Completed | 7/16 [00:06<00:08,  1.09it/s]
Loading safetensors checkpoint shards:  50% Completed | 8/16 [00:07<00:07,  1.09it/s]
Loading safetensors checkpoint shards:  56% Completed | 9/16 [00:08<00:06,  1.09it/s]
Loading safetensors checkpoint shards:  62% Completed | 10/16 [00:08<00:04,  1.35it/s]
Loading safetensors checkpoint shards:  69% Completed | 11/16

(EngineCore_DP0 pid=3507) INFO 04-18 18:04:41 [default_loader.py:268] Loading weights took 14.07 seconds
(EngineCore_DP0 pid=3507) INFO 04-18 18:04:42 [gpu_model_runner.py:2392] Model loading took 56.9342 GiB and 524.967359 seconds
(EngineCore_DP0 pid=3507) INFO 04-18 18:04:49 [backends.py:539] Using cache directory: /home/ubuntu/.cache/vllm/torch_compile_cache/9d5e2a1489/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3507) INFO 04-18 18:04:49 [backends.py:550] Dynamo bytecode transform time: 7.16 s
(EngineCore_DP0 pid=3507) INFO 04-18 18:04:55 [backends.py:194] Cache the graph for dynamic shape for later use
(EngineCore_DP0 pid=3507) INFO 04-18 18:05:43 [backends.py:215] Compiling a graph for dynamic shape takes 53.66 s
(EngineCore_DP0 pid=3507) WARNING 04-18 18:05:45 [fused_moe.py:727] Using default MoE config. Performance might be sub-optimal! Config file not found at ['/home/ubuntu/.venv/lib/python3.10/site-packages/vllm/model_executor/layers/fused_moe/configs/E=128

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:07<00:00,  9.12it/s]


(EngineCore_DP0 pid=3507) INFO 04-18 18:05:56 [gpu_model_runner.py:3118] Graph capturing finished in 8 secs, took 4.77 GiB
(EngineCore_DP0 pid=3507) INFO 04-18 18:05:56 [gpu_worker.py:391] Free memory on device (78.25/79.15 GiB) on startup. Desired GPU memory utilization is (0.9, 71.24 GiB). Actual usage is 56.93 GiB for weight, 1.42 GiB for peak activation, 0.02 GiB for non-torch memory, and 4.77 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=8535829504` to fit into requested memory, or `--kv-cache-memory=16071486464` to fully utilize gpu memory. Current kv cache memory in use is 13814361088 bytes.
(EngineCore_DP0 pid=3507) INFO 04-18 18:05:56 [core.py:218] init engine (profile, create kv cache, warmup model) took 74.07 seconds
INFO 04-18 18:05:57 [llm.py:295] Supported_tasks: ['generate']
INFO 04-18 18:05:57 [__init__.py:36] No IOProcessor plugins requested by the model


In [ ]:
# Промпты 
prompt_template = Template("""
    Сгенерируй описание товара на основе представленных характеристик.

    ХАРАКТЕРИСТИКИ:
    $features

    Строго следуй инструкции по генерации

    ИНСТРУКЦИЯ:
    $instruction

    Также учти, что НЕЛЬЗЯ добавлять никакие дополнительные размышления после параграфов.
    Нужно только ровно одно описание.
    """
)
instruction = """
- Описание должно начинаться с заголовка, выделенного html-тегами <h1> и </h1>
- Все характеристики должны быть учтены в описании
- Под каждую характеристику должен быть выделен отдельный абзац
- каждый абзац должен быть выделен html тегами <p> и </p>
- язык должен быть разнообразным согласно простой метрике лексического разнообразия VocD
- отсутствие повторов
"""

input_data = [prompt_template.substitute(features=str(features), instruction=instruction) for features in result_data]

In [ ]:
# параметры семплирования
sampling_params = SamplingParams(temperature=0.1,
                                top_p=0.9,
                                max_tokens=600,
                                presence_penalty=1.2,
                                repetition_penalty=1,
                                )

def run_generation(input_data, sampling_params, llm):
  start_time = time.time()
  generated_data = llm.generate(input_data, sampling_params)
  end_time = time.time()

  latency = end_time - start_time
  throughput = len(input_data)/latency 

  # В этом случае все ответы мы получаем одновременно,
  # Поэтому можно сказать, что время каждого ответа равно времени полной генерации всех ответов
  # Это основной недостаток офлайн-генерации
  latency_list = [latency for _ in range(len(generated_data))]

  return generated_data, throughput, latency_list


def prepare_results(generated_data, throughput, latency_list):
  generation_results = []
  for output, features, latency in zip(generated_data, result_data, latency_list):
    description = output.outputs[0].text
    score_info = calculate_total_score(output.outputs[0].text, features)

    sample_info = SampleGenerationResult(
        latency=latency,
        description=description,
        total_score=score_info['total_score'],
        detailed_score=copy.deepcopy(score_info['detailed_scores'])
    )
    generation_results.append(sample_info)


  full_generation_result = FullGenerationResult(
      throughput=throughput,
      latency_avg=sum([x.latency for x in generation_results])/len(generation_results),
      total_score_avg=sum([x.total_score for x in generation_results])/len(generation_results),
      outputs=generation_results
  )
  return full_generation_result



generated_data, throughput, latency_list = run_generation(input_data, sampling_params, llm)
print(f"Длительность ответа на единичный запрос {sum(latency_list)/len(latency_list)}")
print(f"Количество запросов, обрабатываемых в секунду {throughput}")


full_generation_result = prepare_results(generated_data, throughput, latency_list)
print_compact_stats(full_generation_result)

Processed prompts: 100%|██████████| 100/100 [00:15<00:00,  6.34it/s, est. speed input: 1994.54 toks/s, output: 2074.78 toks/s]


Длительность ответа на единичный запрос 15.848979949951172
Количество запросов, обрабатываемых в секунду 6.30955432562763
 СТАТИСТИКА ГЕНЕРАЦИИ
   Throughput: 6.31 req/sec
   Avg Latency: 15.849 sec
   Samples: 100
   Avg Score: 50.725

 3 случайных описания:
   1.  <h1>ASOS DESIGN ruched bodice drape maxi dress with wrap waist in red</h1>
    <p>Этот роскошный макси-сарафан от ASOS DESIGN сочетает в себе изысканность и элегантность, подчеркивая женственность с помощью объемного кроя. Удлиненный силуэт и плавные складки создают эффект легкости и грации, идеально подходящий для особых случаев.</p>
    <p>Модель выполнена в ярком красном цвете, который придает образу драматичности и уверенности, привлекая внимание к каждой детали дизайна.</p>
    <p>Особое внимание уделяется передней части — рюшевый боди с мягкими складками добавляет объема и текстурности, подчеркивая линию талии и создавая визуальный акцент на верхней части фигуры.</p>
    <p>Завершающим элементом является пояс с оборот

In [14]:
print("Загрузка модели...")
llm = LLM(
    model="Qwen/Qwen3-30B-A3B-Instruct-2507",
    tensor_parallel_size=1,
    max_model_len=20_000,
    speculative_config={
        "method": "ngram",
        "num_speculative_tokens": 5,
        "prompt_lookup_max": 4,
    },
)

generated_data, throughput, latency_list = run_generation(input_data, sampling_params, llm)
print(f"Длительность ответа на единичный запрос {sum(latency_list)/len(latency_list)}")
print(f"Количество запросов, обрабатываемых в секунду {throughput}")

full_generation_result = prepare_results(generated_data, throughput, latency_list)
print_compact_stats(full_generation_result)

Загрузка модели...
INFO 04-18 18:18:22 [utils.py:328] non-default args: {'max_model_len': 20000, 'disable_log_stats': True, 'speculative_config': {'method': 'ngram', 'num_speculative_tokens': 5, 'prompt_lookup_max': 4}, 'model': 'Qwen/Qwen3-30B-A3B-Instruct-2507'}
INFO 04-18 18:18:23 [__init__.py:742] Resolved architecture: Qwen3MoeForCausalLM
INFO 04-18 18:18:23 [__init__.py:1815] Using max model len 20000
INFO 04-18 18:18:23 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-18 18:18:27 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=4556) INFO 04-18 18:18:28 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=4556) INFO 04-18 18:18:28 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: model='Qwen/Qwen3-30B-A3B-Instruct-2507', speculative_config=SpeculativeConfig(method='ngram', model=None, num_spec_tokens=5), tokenizer='Qwen/Qwen3-30B-A3B-Instruct-2507', skip_tokenizer_init=False, t

[W418 18:18:31.681642711 ProcessGroupNCCL.cpp:981] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=4556) INFO 04-18 18:18:31 [parallel_state.py:1165] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=4556) WARNING 04-18 18:18:31 [topk_topp_sampler.py:69] FlashInfer is not available. Falling back to the PyTorch-native implementation of top-p & top-k sampling. For the best performance, please install FlashInfer.
(EngineCore_DP0 pid=4556) INFO 04-18 18:18:31 [gpu_model_runner.py:2

Loading safetensors checkpoint shards:   0% Completed | 0/16 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   6% Completed | 1/16 [00:00<00:11,  1.26it/s]
Loading safetensors checkpoint shards:  12% Completed | 2/16 [00:01<00:13,  1.01it/s]
Loading safetensors checkpoint shards:  19% Completed | 3/16 [00:03<00:13,  1.06s/it]
Loading safetensors checkpoint shards:  25% Completed | 4/16 [00:04<00:13,  1.08s/it]
Loading safetensors checkpoint shards:  31% Completed | 5/16 [00:05<00:12,  1.10s/it]
Loading safetensors checkpoint shards:  38% Completed | 6/16 [00:06<00:11,  1.11s/it]
Loading safetensors checkpoint shards:  44% Completed | 7/16 [00:07<00:10,  1.12s/it]
Loading safetensors checkpoint shards:  50% Completed | 8/16 [00:08<00:09,  1.13s/it]
Loading safetensors checkpoint shards:  56% Completed | 9/16 [00:09<00:07,  1.13s/it]
Loading safetensors checkpoint shards:  62% Completed | 10/16 [00:10<00:05,  1.06it/s]
Loading safetensors checkpoint shards:  69% Completed | 11/16

(EngineCore_DP0 pid=4556) INFO 04-18 18:18:50 [default_loader.py:268] Loading weights took 17.21 seconds
(EngineCore_DP0 pid=4556) INFO 04-18 18:18:50 [gpu_model_runner.py:2380] Loading drafter model...
(EngineCore_DP0 pid=4556) INFO 04-18 18:18:50 [gpu_model_runner.py:2392] Model loading took 56.9342 GiB and 18.337557 seconds
(EngineCore_DP0 pid=4556) INFO 04-18 18:18:58 [backends.py:539] Using cache directory: /home/ubuntu/.cache/vllm/torch_compile_cache/2f368cd12d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=4556) INFO 04-18 18:18:58 [backends.py:550] Dynamo bytecode transform time: 7.39 s
(EngineCore_DP0 pid=4556) INFO 04-18 18:18:58 [backends.py:194] Cache the graph for dynamic shape for later use
(EngineCore_DP0 pid=4556) INFO 04-18 18:19:03 [backends.py:215] Compiling a graph for dynamic shape takes 5.25 s
(EngineCore_DP0 pid=4556) WARNING 04-18 18:19:05 [fused_moe.py:727] Using default MoE config. Performance might be sub-optimal! Config file not found at ['/h

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:07<00:00,  9.26it/s]


(EngineCore_DP0 pid=4556) INFO 04-18 18:19:30 [gpu_model_runner.py:3118] Graph capturing finished in 8 secs, took 4.77 GiB
(EngineCore_DP0 pid=4556) INFO 04-18 18:19:30 [gpu_worker.py:391] Free memory on device (78.25/79.15 GiB) on startup. Desired GPU memory utilization is (0.9, 71.24 GiB). Actual usage is 56.93 GiB for weight, 1.42 GiB for peak activation, 0.02 GiB for non-torch memory, and 4.77 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=8535829504` to fit into requested memory, or `--kv-cache-memory=16071486464` to fully utilize gpu memory. Current kv cache memory in use is 13812263936 bytes.
(EngineCore_DP0 pid=4556) INFO 04-18 18:19:30 [core.py:218] init engine (profile, create kv cache, warmup model) took 39.76 seconds
INFO 04-18 18:19:31 [llm.py:295] Supported_tasks: ['generate']
INFO 04-18 18:19:31 [__init__.py:36] No IOProcessor plugins requested by the model


Processed prompts: 100%|██████████| 100/100 [00:18<00:00,  5.29it/s, est. speed input: 1663.95 toks/s, output: 1729.03 toks/s]


Длительность ответа на единичный запрос 18.984313249588013
Количество запросов, обрабатываемых в секунду 5.267506845535755
 СТАТИСТИКА ГЕНЕРАЦИИ
   Throughput: 5.27 req/sec
   Avg Latency: 18.984 sec
   Samples: 100
   Avg Score: 50.898

 3 случайных описания:
   1.  <h1>Selected Femme long sleeved shirt in white</h1>
    <p>Эта белоснежная блузка от Selected Femme — идеальный выбор для тех, кто ценит элегантность и комфорт. Ее изысканный дизайн подчеркивает женственность и стиль, делая её универсальным элементом гардероба.</p>
    <p>Доступна в нескольких размерах: EU 34 - UK 6, EU 36 - UK 8, EU 38 - UK 10, EU 40 - UK 12, EU 42 - UK 14, EU 44 - UK 16, что обеспечивает точную посадку для разных фигур.</p>
    <p>Принадлежит к категории классических длиннорукавных блузок, сочетающих практичность с безупречным внешним видом, подходящими как для повседневной носки, так и для деловых встреч.</p>
    <p>Сейчас цена составляет всего 41.50, предлагая превосходное соотношение качества и стоимо

In [ ]:
# Удаление модели и освобождение памяти
# del llm
gc.collect()

# Квантованная модель
model_name = "cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit"

# Инициализация модели
print("Загрузка модели...")
llm = LLM(
    model=model_name,
    tensor_parallel_size=1,
    max_model_len=20_000
)

generated_data, throughput, latency_list = run_generation(input_data, sampling_params, llm)
print(f"Длительность ответа на единичный запрос {sum(latency_list)/len(latency_list)}")
print(f"Количество запросов, обрабатываемых в секунду {throughput}")

full_generation_result = prepare_results(generated_data, throughput, latency_list)
print_compact_stats(full_generation_result)

Загрузка модели...
INFO 04-18 19:14:32 [utils.py:328] non-default args: {'max_model_len': 20000, 'disable_log_stats': True, 'model': 'cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit'}
INFO 04-18 19:14:39 [__init__.py:742] Resolved architecture: Qwen3MoeForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 04-18 19:14:39 [__init__.py:1815] Using max model len 20000
WARNING 04-18 19:14:41 [_ipex_ops.py:16] Import error msg: No module named 'intel_extension_for_pytorch'


2026-04-18 19:14:41,421	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 04-18 19:14:42 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 04-18 19:14:43 [__init__.py:2974] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 04-18 19:14:46 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=11914) INFO 04-18 19:14:48 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=11914) INFO 04-18 19:14:48 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: model='cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit', speculative_config=None, tokenizer='cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=20000, download_dir=None, load_format=auto

[W418 19:14:49.494530740 ProcessGroupNCCL.cpp:981] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=11914) INFO 04-18 19:14:49 [parallel_state.py:1165] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=11914) WARNING 04-18 19:14:49 [topk_topp_sampler.py:69] FlashInfer is not available. Falling back to the PyTorch-native implementation of top-p & top-k sampling. For the best performance, please install FlashInfer.
(EngineCore_DP0 pid=11914) INFO 04-18 19:14:50 [gpu_model_runner.p

(EngineCore_DP0 pid=11914) /home/ubuntu/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:805: UserWarning: Not enough free disk space to download the file. The expected file size is: 5001.28 MB. The target location /home/ubuntu/.cache/huggingface/hub/models--cpatonn--Qwen3-30B-A3B-Instruct-2507-AWQ-4bit/blobs only has 647.00 MB free disk space.
(EngineCore_DP0 pid=11914)   warnings.warn(
(EngineCore_DP0 pid=11914) /home/ubuntu/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:805: UserWarning: Not enough free disk space to download the file. The expected file size is: 5001.71 MB. The target location /home/ubuntu/.cache/huggingface/hub/models--cpatonn--Qwen3-30B-A3B-Instruct-2507-AWQ-4bit/blobs only has 647.00 MB free disk space.
(EngineCore_DP0 pid=11914)   warnings.warn(
Cancellation requested; stopping current tasks.
[rank0]:[W418 19:34:15.212608094 ProcessGroupNCCL.cpp:1538] Warning: WARNING: destroy_process_group() was not called before program 

In [ ]:
# Удаление модели и освобождение памяти
#del llm
gc.collect()

# Дополнительная очистка GPU памяти (если используется CUDA)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

## Онлайн генерация

In [ ]:
# Обработчики запросов
nest_asyncio.apply()

async def send_to_generation(prompt, model_name="Qwen/Qwen3-30B-A3B-Instruct-2507", sampling_params=None):
    base_url = "http://localhost:8000/v1"
    params = copy.deepcopy(sampling_params) or {}
    params["model_name"] = model_name
    params["prompt"] = prompt
    async with aiohttp.ClientSession() as session:
        start_time = time.time()
        print(f"start {time.strftime('%H:%M:%S')} - {prompt[:20]}...")
        async with session.post(
            f"{base_url}/completions",
            json={
                "model": model_name,
                "prompt": prompt,
                "max_tokens": 500,
                "temperature": 0.7
            },
            headers={"Authorization": "Bearer token-abc123"},
            timeout=100
        ) as response:
            result = await response.json()
            end_time = time.time()
            print(f"end {time.strftime('%H:%M:%S')} - {prompt[:20]}... (duration: {end_time-start_time:.5f}s)")
            return result, start_time, end_time

async def schedule_requests(prompts, pause_duration=1, sampling_params=None, model_name="Qwen/Qwen3-30B-A3B-Instruct-2507"):
    tasks = []

    for i, prompt in enumerate(prompts):
        # Создаём задачу, но не ждём её завершения
        task = asyncio.create_task(send_to_generation(prompt, model_name, sampling_params))
        tasks.append(task)

        # Ждём pause_duration секунд перед следующим запросом
        if i < len(prompts) - 1:  # Не ждём после последнего запроса
            await asyncio.sleep(pause_duration)

    # Ждём завершения всех задач
    results = await asyncio.gather(*tasks)
    return results

In [4]:
def prepare_results_online(results):
    generated_data = [res[0] for res in results]
    latency_list = [res[2]-res[1] for res in results] # вычислите latency для каждого запроса
    throughput = len(results)/(results[-1][2] - results[0][1]) # вычислите пропускную способность

    generation_results = []
    for output, features, latency in zip(generated_data, result_data, latency_list):
        description = output['choices'][0]['text']
        score_info = calculate_total_score(output['choices'][0]['text'], features)

        sample_info = SampleGenerationResult(
            latency=latency,
            description=description,
            total_score=score_info['total_score'],
            detailed_score=copy.deepcopy(score_info['detailed_scores'])
        )
        generation_results.append(sample_info)

    full_generation_result = FullGenerationResult(
        throughput=throughput,
        latency_avg=sum([x.latency for x in generation_results])/len(generation_results), # вычислите среднее latency
        total_score_avg=sum([x.total_score for x in generation_results])/len(generation_results), # вычислите средний скор по заданной метрике
        outputs=generation_results
    )
    return full_generation_result


In [ ]:
# Полновесная модель
results = await schedule_requests(input_data, pause_duration=4)
full_generation_result = prepare_results_online(results)
print_compact_stats(full_generation_result)

start 18:57:13 - 
    Сгенерируй опис...
end 18:57:17 - 
    Сгенерируй опис... (duration: 3.74423s)
start 18:57:17 - 
    Сгенерируй опис...
end 18:57:20 - 
    Сгенерируй опис... (duration: 2.95535s)
start 18:57:21 - 
    Сгенерируй опис...
end 18:57:25 - 
    Сгенерируй опис... (duration: 3.12505s)
start 18:57:25 - 
    Сгенерируй опис...
end 18:57:28 - 
    Сгенерируй опис... (duration: 2.75732s)
start 18:57:29 - 
    Сгенерируй опис...
start 18:57:33 - 
    Сгенерируй опис...
end 18:57:34 - 
    Сгенерируй опис... (duration: 4.74779s)
end 18:57:37 - 
    Сгенерируй опис... (duration: 3.94382s)
start 18:57:38 - 
    Сгенерируй опис...
end 18:57:41 - 
    Сгенерируй опис... (duration: 3.39464s)
start 18:57:42 - 
    Сгенерируй опис...
end 18:57:45 - 
    Сгенерируй опис... (duration: 3.53851s)
start 18:57:46 - 
    Сгенерируй опис...
end 18:57:48 - 
    Сгенерируй опис... (duration: 2.62213s)
start 18:57:50 - 
    Сгенерируй опис...
end 18:57:51 - 
    Сгенерируй опис... (duration: 

In [ ]:
# Квантованная модель
results = await schedule_requests(input_data, pause_duration=4, model_name="cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit")
full_generation_result = prepare_results_online(results)
print_compact_stats(full_generation_result)

In [ ]:
# Простой запуск vllm

vllm serve "Qwen/Qwen3-30B-A3B-Instruct-2507" \
  --dtype auto \
  --api-key token-abc123 \
  --max-model-len '20k' 